# When the calculator is wrong

Three ways to get a false leftover — and how this calculator refuses two of them, and disagrees with the third.

Run from the package root, or from this `jupyter_notebooks/` folder.

In [1]:
from pathlib import Path
import sys

p = Path.cwd().resolve()
if p.name == "jupyter_notebooks":
    root = p.parent
elif (p / "code" / "finite_obstruction.py").is_file():
    root = p
elif (p / "coda_atlas" / "F005_coda" / "foc" / "code").is_dir():
    root = p / "coda_atlas" / "F005_coda" / "foc"
elif (p / "foc" / "code").is_dir():
    root = p / "foc"
else:
    root = p
assert (root / "code").is_dir(), "run from the package root or jupyter_notebooks/"
if str(root / "code") not in sys.path:
    sys.path.insert(0, str(root / "code"))

from sympy import Matrix, Rational
from finite_obstruction import FiniteObstructionPacket, quotient_certificate

## Least squares is the wrong distance

Suppose licensed repairs are “add any multiple of $(1,1,1)$”, and the vector is $(0,0,10)$. You are trying to cancel as much as possible in $\ell^1$ (sum of absolute values).

Least squares ($\ell^2$) projects onto the diagonal at $t=10/3$. The $\ell^1$ size of *that* residual is $40/3$. The true $\ell^1$ leftover is $10$, attained by adding nothing ($t=0$). Using the wrong distance quietly changes the answer.

In [2]:
cert = quotient_certificate(Matrix([[1], [1], [1]]), [0, 0, 10], [1, 1, 1])
l2_as_l1 = Rational(40, 3)
print("L1 leftover:", cert.value)
print("L1 size of the L2 residual:", l2_as_l1)
assert cert.value == Rational(10)
assert cert.value != l2_as_l1
assert cert.duality_gap == 0

L1 leftover: 10
L1 size of the L2 residual: 40/3


## A zero weight is refused

The leftover is a *norm* only if every coordinate costs something. If a coordinate has weight $0$, leftover can hide there for free, and “distance to the repair subspace” is no longer a norm. The calculator refuses rather than return a number.

In [3]:
try:
    quotient_certificate(Matrix.zeros(2, 0), [0, 1], [1, 0])
except ValueError as err:
    print("refused:", err)
else:
    raise AssertionError("zero weight must be refused")

refused: quotient weights must be strictly positive


## A broken next-defect map is refused

The two stages have to agree: an exact first-stage repair must produce no next-stage defect. That is the cochain rule $d_1 d_0 = 0$. If $d_0$ is “copy the number” and $d_1$ is also “copy the number”, then $d_1 d_0 = 1 \neq 0$: an exact repair *creates* a next-stage defect. The setup is inconsistent, so the calculator refuses before any diagnosis.

In [4]:
try:
    FiniteObstructionPacket.create(
        d0=[[1]],
        d1=[[1]],
        a2=Matrix.zeros(1, 0),
        w1=[1],
        w2=[1],
    )
except ValueError as err:
    print("refused:", err)
else:
    raise AssertionError("broken d1 d0 must be refused")

refused: cochain condition d1*d0=0 failed
